# Assembling the master modeling dataset 

To-do: 
- Merge EJI, age, county shapefiles, and distances together into one big GDF 

In [1]:
import pandas as pd
import requests
import geopandas as gpd
from shapely.geometry import Polygon, Point
import numpy as np

In [2]:
import os
os.chdir('/users/bkung/fooddesertproject')

In [3]:
# reading in EJI data file from CSV 
EJI_data = pd.read_csv('modeling_data/EJI_2024_United_States.csv')

In [4]:
# filtering data to relevant counties: Bexar, Dallas, Tarrant, Travis, Harris
EJI_texas = EJI_data[EJI_data['STATEFP'] == 48]
EJI_relevant_counties = EJI_texas[EJI_texas['COUNTYFP'].isin([29, 453, 201, 113, 439])] # Bexar, Travis, Harris, Dallas, Tarrant

In [5]:
# filtering to relevant columns 
modeling_EJI_data = EJI_relevant_counties[
    ['COUNTY', 'GEOID', 'TRACTCE', 'E_WLKIND', 'EPL_WLKIND', 'E_TOTPOP', 
     'M_TOTPOP', 'E_POV200', 'E_NOHSDP', 'E_UNINSUR', 'EPL_CHD', 
     'E_DIABETES', 'E_AFAM','E_ASIAN', 'E_HISP']
    ]

In [6]:
# RETRIEVING ACS MEDIAN AGE DATA FROM API

# 1. Setup exact parameters
API_KEY = "3260419d0dd1c45a470edaf688621f89b71fa441"
STATE_FIPS = "48"    # Texas
COUNTY_LIST = ["029", "453", "201", "113", "439"] 

# Using the Data Profile endpoint and its matching Median Age variable
# DP05_0018E = Median Age (Total Population)
URL_BASE = "https://api.census.gov/data/2024/acs/acs5/profile"
VARIABLES = "NAME,DP05_0018E"

all_tracts_data = []

for county_fips in COUNTY_LIST:
    # Build URL carefully
    url = f"{URL_BASE}?get={VARIABLES}&for=tract:*&in=state:{STATE_FIPS}+county:{county_fips}&key={API_KEY}"
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        # SAFETY CHECK: Verify the API returned JSON text before decoding it
        content_type = response.headers.get('Content-Type', '')
        if 'application/json' in content_type:
            json_data = response.json()
            
            headers = json_data[0]
            rows = json_data[1:]
            county_df = pd.DataFrame(rows, columns=headers)
            all_tracts_data.append(county_df)
        else:
            # If the Census Bureau sent a plain text error, print it out clearly!
            print(f"\n[Census API Error] Server returned text instead of data for county {county_fips}:")
            print(response.text.strip())

    except requests.exceptions.RequestException as e:
        print(f"Network or HTTP error for county {county_fips}: {e}")

# 3. Combine results safely
if all_tracts_data:
    final_df = pd.concat(all_tracts_data, ignore_index=True)
    
    # Convert and clean columns
    final_df["DP05_0018E"] = pd.to_numeric(final_df["DP05_0018E"], errors='coerce')
    final_df = final_df.rename(columns={"DP05_0018E": "median_age"})
    
    print("\n--- SUCCESS! First 5 rows: ---")
    print(final_df[["NAME", "tract", "median_age"]].head())
else:
    print("\nNo data was collected. Read the API text errors above to see why.")



--- SUCCESS! First 5 rows: ---
                                     NAME   tract  median_age
0  Census Tract 1101; Bexar County; Texas  110100        34.6
1  Census Tract 1103; Bexar County; Texas  110300        37.6
2  Census Tract 1105; Bexar County; Texas  110500        25.4
3  Census Tract 1106; Bexar County; Texas  110600        37.2
4  Census Tract 1107; Bexar County; Texas  110700        47.3


In [7]:
# reecoding missing values 
final_df['median_age'] = final_df['median_age'].replace({-666666666.0: np.nan})

In [8]:
# converting 'tract' to int 
final_df['tract'] = final_df['tract'].astype('Int64')

In [9]:
# renaming columns 
modeling_EJI_data = modeling_EJI_data.rename(
    columns={'TRACTCE': 'tract', 'E_WLKIND': 'walking_ind', 'EPL_WLKIND':'walkind_inv_perc', 'E_TOTPOP': 'total_pop', 'M_TOTPOP': 'total_pop_moe', 
             'E_POV200': 'below_200_fed_poverty_percentage', 'E_NOHSDP':'no_hs_diploma', 'E_UNINSUR': 'uninsured'})


In [12]:
# merging median age and EJI data 
EJI_age = pd.merge(modeling_EJI_data, final_df, how='left', on='tract')
EJI_age = EJI_age.drop(labels=['NAME', 'state', 'county'], axis=1)

In [ ]:
EJI_age.head()

In [43]:
# splitting them up to facilitate merges with SNAP data 
bexar_EJI_age = EJI_age[EJI_age['COUNTY'] == 'Bexar County']
travis_EJI_age = EJI_age[EJI_age['COUNTY'] == 'Travis County']
harris_EJI_age = EJI_age[EJI_age['COUNTY'] == 'Harris County']
dallas_EJI_age = EJI_age[EJI_age['COUNTY'] == 'Dallas County']
tarrant_EJI_age = EJI_age[EJI_age['COUNTY'] == 'Tarrant County']

In [44]:
# reading in SNAP data
bexar_snap = pd.read_csv('Bexar_Data/cleaned_bexar_snap_data.csv')
travis_snap = pd.read_csv('Travis_Data/cleaned_travis_snap_data.csv')
harris_snap = pd.read_csv('Harris_Data/cleaned_harris_snap_data.csv')
dallas_snap = pd.read_csv('Dallas_Data/cleaned_dallas_snap_data.csv')
tarrant_snap = pd.read_csv('Tarrant_Data/cleaned_tarrant_snap_data.csv')

In [45]:
# separating farmers markets
bexar_fm = bexar_snap[bexar_snap['Store_Type'] == 'Farmers and Markets']
bexar_grocery= bexar_snap[bexar_snap['Store_Type'] != 'Farmers and Markets']
travis_fm = travis_snap[travis_snap['Store_Type'] == 'Farmers and Markets']
travis_grocery = travis_snap[travis_snap['Store_Type'] != 'Farmers and Markets']
harris_fm = harris_snap[harris_snap['Store_Type'] == 'Farmers and Markets']
harris_grocery = harris_snap[harris_snap['Store_Type'] != 'Farmers and Markets']
dallas_fm = dallas_snap[dallas_snap['Store_Type'] == 'Farmers and Markets']
dallas_grocery = dallas_snap[dallas_snap['Store_Type'] != 'Farmers and Markets']
tarrant_fm = tarrant_snap[tarrant_snap['Store_Type'] == 'Farmers and Markets']
tarrant_grocery = tarrant_snap[tarrant_snap['Store_Type'] != 'Farmers and Markets']

In [46]:
# converting to gdfs 
bexar_fm_gdf = gpd.GeoDataFrame(bexar_fm, geometry=gpd.points_from_xy(bexar_fm.Longitude, bexar_fm.Latitude), crs='EPSG:4326')
bexar_grocery_gdf = gpd.GeoDataFrame(bexar_grocery, geometry=gpd.points_from_xy(bexar_grocery.Longitude, bexar_grocery.Latitude), crs='EPSG:4326')

travis_fm_gdf = gpd.GeoDataFrame(travis_fm, geometry=gpd.points_from_xy(travis_fm.Longitude, travis_fm.Latitude), crs='EPSG:4326')
travis_grocery_gdf = gpd.GeoDataFrame(travis_grocery, geometry=gpd.points_from_xy(travis_grocery.Longitude, travis_grocery.Latitude), crs='EPSG:4326')

harris_fm_gdf = gpd.GeoDataFrame(harris_fm, geometry=gpd.points_from_xy(harris_fm.Longitude, harris_fm.Latitude), crs='EPSG:4326')
harris_grocery_gdf = gpd.GeoDataFrame(harris_grocery, geometry=gpd.points_from_xy(harris_grocery.Longitude, harris_grocery.Latitude), crs='EPSG:4326')

dallas_fm_gdf = gpd.GeoDataFrame(dallas_fm, geometry=gpd.points_from_xy(dallas_fm.Longitude, dallas_fm.Latitude), crs='EPSG:4326')
dallas_grocery_gdf = gpd.GeoDataFrame(dallas_grocery, geometry=gpd.points_from_xy(dallas_grocery.Longitude, dallas_grocery.Latitude), crs='EPSG:4326')

tarrant_fm_gdf = gpd.GeoDataFrame(tarrant_fm, geometry=gpd.points_from_xy(tarrant_fm.Longitude, tarrant_fm.Latitude), crs='EPSG:4326')
tarrant_grocery_gdf = gpd.GeoDataFrame(tarrant_grocery, geometry=gpd.points_from_xy(tarrant_grocery.Longitude, tarrant_grocery.Latitude), crs='EPSG:4326')

In [52]:
tarrant_grocery_gdf['Store_Type'].value_counts()

Store_Type
Super Store      130
Grocery Store     97
Supermarket       96
Other             43
Name: count, dtype: int64

In [96]:
# Reading in tract shapefiles
tracts_shp = gpd.read_file("food_justice/bexar_county/tl_2024_48_tract.shp")

# projecting to distance-friendly CRS 
bexar_tracts = tracts_shp[tracts_shp["COUNTYFP"] == "029"]
bexar_tracts_projected = bexar_tracts.to_crs('EPSG:2278') 

travis_tracts = tracts_shp[tracts_shp["COUNTYFP"] == "453"]
travis_tracts_projected = travis_tracts.to_crs('EPSG:2277') 

harris_tracts = tracts_shp[tracts_shp["COUNTYFP"] == "201"]
harris_tracts_projected = harris_tracts.to_crs('EPSG:2277') 

dallas_tracts = tracts_shp[tracts_shp["COUNTYFP"] == "113"]
dallas_tracts_projected = dallas_tracts.to_crs('EPSG:2276') 

tarrant_tracts = tracts_shp[tracts_shp["COUNTYFP"] == "439"]
tarrant_tracts_projected = tarrant_tracts.to_crs('EPSG:2276') 

# doing the same for the farmers markets/grocery stores
bexar_fm_gdf = bexar_fm_gdf.to_crs('EPSG:2278')
bexar_grocery_gdf = bexar_grocery_gdf.to_crs('EPSG:2278')

travis_fm_gdf = travis_fm_gdf.to_crs('EPSG:2277')
travis_grocery_gdf = travis_grocery_gdf.to_crs('EPSG:2277')

harris_fm_gdf = harris_fm_gdf.to_crs('EPSG:2277')
harris_grocery_gdf = harris_grocery_gdf.to_crs('EPSG:2277')

dallas_fm_gdf = dallas_fm_gdf.to_crs('EPSG:2276')
dallas_grocery_gdf = dallas_grocery_gdf.to_crs('EPSG:2276')

tarrant_fm_gdf = tarrant_fm_gdf.to_crs('EPSG:2276')
tarrant_grocery_gdf = tarrant_grocery_gdf.to_crs('EPSG:2276')

# Computing centroids 
bexar_tracts_projected["centroid"] = bexar_tracts_projected.geometry.centroid

travis_tracts_projected["centroid"] = travis_tracts_projected.geometry.centroid

harris_tracts_projected["centroid"] = harris_tracts_projected.geometry.centroid

dallas_tracts_projected["centroid"] = dallas_tracts_projected.geometry.centroid

tarrant_tracts_projected["centroid"] = tarrant_tracts_projected.geometry.centroid

# 3. Creating new GDFs with centroids as main geometry 
bexar_centroids_projected = bexar_tracts_projected.set_geometry("centroid")
travis_centroids_projected = travis_tracts_projected.set_geometry("centroid")
harris_centroids_projected= harris_tracts_projected.set_geometry("centroid")
dallas_centroids_projected = dallas_tracts_projected.set_geometry("centroid")
tarrant_centroids_projected = tarrant_tracts_projected.set_geometry("centroid")

# 4. Find the nearest point and calculate the distance
# 'distance_col' automatically creates a column with the exact distance
bexar_nearest_fm = gpd.sjoin_nearest(
    bexar_centroids_projected, 
    bexar_fm_gdf, 
    how="left", 
    distance_col="distance_to_nearest_fm"
)

bexar_nearest_grocery = gpd.sjoin_nearest(
    bexar_centroids_projected, 
    bexar_grocery_gdf, 
    how="left", 
    distance_col="distance_to_nearest_grocery"
)

travis_nearest_fm = gpd.sjoin_nearest(
    travis_centroids_projected, 
    travis_fm_gdf, 
    how="left", 
    distance_col="distance_to_nearest_fm"
)

travis_nearest_grocery = gpd.sjoin_nearest(
    travis_centroids_projected, 
    travis_grocery_gdf, 
    how="left", 
    distance_col="distance_to_nearest_grocery"
)

harris_nearest_fm = gpd.sjoin_nearest(
    harris_centroids_projected, 
    harris_fm_gdf, 
    how="left", 
    distance_col="distance_to_nearest_fm"
)

harris_nearest_grocery = gpd.sjoin_nearest(
    harris_centroids_projected, 
    harris_grocery_gdf, 
    how="left", 
    distance_col="distance_to_nearest_grocery"
)

dallas_nearest_fm = gpd.sjoin_nearest(
    dallas_centroids_projected, 
    dallas_fm_gdf, 
    how="left", 
    distance_col="distance_to_nearest_fm"
)

dallas_nearest_grocery = gpd.sjoin_nearest(
    dallas_centroids_projected, 
    dallas_grocery_gdf, 
    how="left", 
    distance_col="distance_to_nearest_grocery"
)

tarrant_nearest_fm = gpd.sjoin_nearest(
    tarrant_centroids_projected, 
    tarrant_fm_gdf, 
    how="left", 
    distance_col="distance_to_nearest_fm"
)

tarrant_nearest_grocery = gpd.sjoin_nearest(
    tarrant_centroids_projected, 
    tarrant_grocery_gdf, 
    how="left", 
    distance_col="distance_to_nearest_grocery"
)

In [102]:
# merging farmers market and grocery stores 
bexar_distances = bexar_nearest_grocery.merge(bexar_nearest_fm, how='left', on="TRACTCE", suffixes=("_drop", ""))
bexar_distances = bexar_distances.drop(columns=[col for col in bexar_distances.columns if col.endswith("_drop")])

travis_distances = travis_nearest_grocery.merge(travis_nearest_fm, how='left', on="TRACTCE", suffixes=("_drop", ""))
travis_distances = travis_distances.drop(columns=[col for col in travis_distances.columns if col.endswith("_drop")])

harris_distances = harris_nearest_grocery.merge(harris_nearest_fm, how='left', on="TRACTCE", suffixes=("_drop", ""))
harris_distances = harris_distances.drop(columns=[col for col in harris_distances.columns if col.endswith("_drop")])

dallas_distances = dallas_nearest_grocery.merge(dallas_nearest_fm, how='left', on="TRACTCE", suffixes=("_drop", ""))
dallas_distances = dallas_distances.drop(columns=[col for col in dallas_distances.columns if col.endswith("_drop")])

tarrant_distances = tarrant_nearest_grocery.merge(tarrant_nearest_fm, how='left', on="TRACTCE", suffixes=("_drop", ""))
tarrant_distances = tarrant_distances.drop(columns=[col for col in tarrant_distances.columns if col.endswith("_drop")])

In [119]:
# cleaning up columns
bexar_distances = bexar_distances[['TRACTCE', 'COUNTYFP', 'County', 'GEOID', 'ALAND', 'AWATER', 'distance_to_nearest_grocery', 'distance_to_nearest_fm']]
travis_distances = travis_distances[['TRACTCE', 'COUNTYFP', 'County', 'GEOID', 'ALAND', 'AWATER', 'distance_to_nearest_grocery', 'distance_to_nearest_fm']]
harris_distances = harris_distances[['TRACTCE', 'COUNTYFP', 'County', 'GEOID', 'ALAND', 'AWATER', 'distance_to_nearest_grocery', 'distance_to_nearest_fm']]
dallas_distances = dallas_distances[['TRACTCE', 'COUNTYFP', 'County', 'GEOID', 'ALAND', 'AWATER', 'distance_to_nearest_grocery', 'distance_to_nearest_fm']]
tarrant_distances = tarrant_distances[['TRACTCE', 'COUNTYFP', 'County', 'GEOID', 'ALAND', 'AWATER', 'distance_to_nearest_grocery', 'distance_to_nearest_fm']]